# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip -q install duckdb pandas scikit-learn

import getpass

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

hf_token = getpass.getpass("Enter your Hugging Face read token: ")
if not hf_token:
    raise RuntimeError("A Hugging Face read token is required to access the gated warehouse.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
del hf_token

release = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{release}/fact_content_daily_performance/month=2026-03/*.parquet"
month_source = f"read_parquet('{month_path}')"

FEATURE_DEFINITIONS = {
    "avg_daily_impressions_pre23": "Mean daily GSC impressions from March 1-23; available before the March 24 decision.",
    "total_clicks_pre23": "Total GSC clicks from March 1-23; available before the decision.",
    "ctr_pre23_pct": "Clicks / impressions * 100 from March 1-23; available before the decision.",
    "avg_position_pre23": "Mean positive GSC position from March 1-23; available before the decision.",
    "active_search_days_pre23": "Days with positive GSC impressions from March 1-23; available before the decision.",
}
FEATURE_COLUMNS = list(FEATURE_DEFINITIONS)
TARGET_COLUMN = "future_decline"

audit_window_start = "2026-03-01"
feature_window_end = "2026-03-23"
decision_date = "2026-03-24"
reference_window_start = "2026-03-16"
outcome_window_end = "2026-03-31"

feature_sql = f"""
WITH daily AS (
    SELECT CAST(report_date AS DATE) AS report_date,
           client_hash_id,
           content_hash_id,
           TRY_CAST(gsc_impressions AS DOUBLE) AS impressions,
           TRY_CAST(gsc_clicks AS DOUBLE) AS clicks,
           TRY_CAST(gsc_avg_position AS DOUBLE) AS avg_position
    FROM {month_source}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
), page_windows AS (
    SELECT client_hash_id,
           content_hash_id,
           AVG(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
           ) AS avg_daily_impressions_pre23,
           SUM(clicks) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
           ) AS total_clicks_pre23,
           100.0 * SUM(clicks) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
           ) / NULLIF(SUM(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
           ), 0) AS ctr_pre23_pct,
           AVG(NULLIF(avg_position, 0)) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
           ) AS avg_position_pre23,
           COUNT(DISTINCT report_date) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-23'
                 AND impressions > 0
           ) AS active_search_days_pre23,
           COUNT(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-23'
           ) AS reference_observed_days,
           COUNT(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-24' AND DATE '2026-03-31'
           ) AS outcome_observed_days,
           AVG(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-23'
           ) AS reference_avg_daily_impressions,
           AVG(impressions) FILTER (
               WHERE report_date BETWEEN DATE '2026-03-24' AND DATE '2026-03-31'
           ) AS outcome_avg_daily_impressions
    FROM daily
    GROUP BY client_hash_id, content_hash_id
)
SELECT *,
       outcome_avg_daily_impressions / NULLIF(reference_avg_daily_impressions, 0)
           AS future_change_ratio,
       CASE WHEN outcome_avg_daily_impressions < 0.8 * reference_avg_daily_impressions
            THEN 1 ELSE 0 END AS future_decline
FROM page_windows
WHERE reference_observed_days >= 3
  AND outcome_observed_days >= 3
  AND reference_avg_daily_impressions > 0
"""

feature_rows = con.sql(feature_sql).df()
if feature_rows.empty:
    raise ValueError(
        "No client-pages meet the minimum observed-day rule. Review March data coverage "
        "before changing the thresholds."
    )

feature_frame = feature_rows[FEATURE_COLUMNS].copy()
target = feature_rows[TARGET_COLUMN].astype("int8")
client_groups = feature_rows["client_hash_id"].copy()
leak_ratio = feature_rows["future_change_ratio"].copy()

forbidden_features = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "reference_avg_daily_impressions",
    "outcome_avg_daily_impressions",
    "future_change_ratio",
    "future_decline",
}
assert forbidden_features.isdisjoint(FEATURE_COLUMNS)
assert len(feature_frame) == len(target) == len(client_groups)

print(f"One row = one eligible client-page; rows: {len(feature_frame):,}")
print(f"Client groups: {client_groups.nunique():,}; future-decline proxy prevalence: {target.mean():.1%}")
print("Feature window: March 1-23; decision point: March 24; reference: March 16-23; outcome: March 24-31.")
print("Identifiers are retained only for grouping; the preview below contains no identifiers.")
display(feature_frame.assign(future_decline=target).head(5))

Enter your Hugging Face read token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

One row = one eligible client-page; rows: 122,789
Client groups: 43; future-decline proxy prevalence: 26.7%
Feature window: March 1-23; decision point: March 24; reference: March 16-23; outcome: March 24-31.
Identifiers are retained only for grouping; the preview below contains no identifiers.


,avg_daily_impressions_pre23,total_clicks_pre23,ctr_pre23_pct,avg_position_pre23,active_search_days_pre23,future_decline
0,239.130435,6.0,0.109091,7.545698,23,1
1,15.043478,0.0,0.000000,3.527570,23,0
2,207.043478,3.0,0.062999,6.486842,23,0
3,169.391304,12.0,0.308008,7.469719,23,1
4,15.869565,1.0,0.273973,3.530152,23,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning / missingness | Available when? |
|---|---|---|
| `avg_daily_impressions_pre23` | Mean daily impressions; nullable when no observed impressions in that feature window. | By the March 24 decision, from March 1–23. |
| `total_clicks_pre23` | Total clicks; nullable if the window has no non-null clicks. | By the decision, from March 1–23. |
| `ctr_pre23_pct` | Weighted clicks / impressions × 100; null when the impression denominator is zero. | By the decision, using only March 1–23. |
| `avg_position_pre23` | Mean positive position; zero is treated as no position and excluded. | By the decision, using only March 1–23. |
| `active_search_days_pre23` | Count of days with positive impressions. | By the decision, from dates through March 23. |

All five are numeric. Missing values remain missing in the feature frame and are imputed using training-fold medians only. Require at least three observed days in both the 8-day reference window and the 8-day outcome window; this eligibility rule uses outcome-period coverage and must be disclosed as a population-selection limitation, not a feature.

In [2]:
feature_notes = pd.DataFrame(
    [
        {
            "feature": feature,
            "meaning_and_missingness": definition,
            "missing_rows": int(feature_frame[feature].isna().sum()),
            "missing_rate_pct": round(float(feature_frame[feature].isna().mean() * 100), 2),
            "available_at_decision": "Yes, only March 1-23 values; assuming data is published by March 24.",
            "type": "numeric",
        }
        for feature, definition in FEATURE_DEFINITIONS.items()
    ]
)
print("Feature meanings, missingness, and decision-time availability:")
display(feature_notes)
print("No categorical predictors are used; this compact leakage check uses five numeric features.")

Feature meanings, missingness, and decision-time availability:


,feature,meaning_and_missingness,missing_rows,missing_rate_pct,available_at_decision,type
0,avg_daily_impressions_pre23,Mean daily GSC impressions from March 1-23; av...,0,0.00,"Yes, only March 1-23 values; assuming data is ...",numeric
1,total_clicks_pre23,Total GSC clicks from March 1-23; available be...,0,0.00,"Yes, only March 1-23 values; assuming data is ...",numeric
2,ctr_pre23_pct,Clicks / impressions * 100 from March 1-23; av...,0,0.00,"Yes, only March 1-23 values; assuming data is ...",numeric
3,avg_position_pre23,Mean positive GSC position from March 1-23; av...,22,0.02,"Yes, only March 1-23 values; assuming data is ...",numeric
4,active_search_days_pre23,Days with positive GSC impressions from March ...,0,0.00,"Yes, only March 1-23 values; assuming data is ...",numeric


No categorical predictors are used; this compact leakage check uses five numeric features.


## 3. The leakage hunt

The label is `future_decline = 1` when March 24–31 average daily impressions are below 80% of March 16–23 average daily impressions. The deliberately leaky candidate, `future_change_ratio`, is computed from those same future outcome impressions and directly defines the label threshold. It is forbidden as a feature.

Compare a compact decision tree using only the five clean pre-decision features with the same tree trained on `future_change_ratio` alone. Both runs use one client-held-out split, the same labels, and the same ranking metrics. The leaked score is a diagnostic demonstration only; after printing the comparison, delete the leaked frame/score variables and retain the clean score as the only valid result.

In [3]:
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

split_indices = None
for split_seed in (42, 17, 11, 7, 23):
    candidate_split = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=split_seed).split(
            feature_frame, target, groups=client_groups
        )
    )
    candidate_train, candidate_test = candidate_split
    if target.iloc[candidate_train].nunique() == 2 and target.iloc[candidate_test].nunique() == 2:
        split_indices = candidate_split
        break
if split_indices is None:
    raise ValueError("Could not create a client-held-out split with both target classes.")

train_indices, test_indices = split_indices
X_train = feature_frame.iloc[train_indices]
X_test = feature_frame.iloc[test_indices]
y_train = target.iloc[train_indices]
y_test = target.iloc[test_indices]
train_clients = set(client_groups.iloc[train_indices])
test_clients = set(client_groups.iloc[test_indices])
assert train_clients.isdisjoint(test_clients)

clean_model = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
)
clean_model.fit(X_train, y_train)
clean_scores = clean_model.predict_proba(X_test)[:, list(clean_model.classes_).index(1)]

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=1, random_state=42),
)
leaky_train = leak_ratio.iloc[train_indices].to_frame(name="future_change_ratio")
leaky_test = leak_ratio.iloc[test_indices].to_frame(name="future_change_ratio")
leaky_model.fit(leaky_train, y_train)
leaky_scores = leaky_model.predict_proba(leaky_test)[:, list(leaky_model.classes_).index(1)]


def precision_at_k(y_true, scores, k=50):
    truth = np.asarray(y_true, dtype=int)
    ranking = np.asarray(scores, dtype=float)
    actual_k = min(k, len(truth))
    if actual_k == 0:
        return np.nan
    top = np.argsort(-ranking, kind="mergesort")[:actual_k]
    return float(truth[top].mean())

leakage_results = pd.DataFrame(
    [
        {
            "feature_set": "Clean: five pre-decision features",
            "test_pages": len(y_test),
            "test_clients": len(test_clients),
            "target_base_rate": float(y_test.mean()),
            "average_precision": average_precision_score(y_test, clean_scores),
            "precision_at_50": precision_at_k(y_test, clean_scores, 50),
        },
        {
            "feature_set": "INVALID: future_change_ratio leakage",
            "test_pages": len(y_test),
            "test_clients": len(test_clients),
            "target_base_rate": float(y_test.mean()),
            "average_precision": average_precision_score(y_test, leaky_scores),
            "precision_at_50": precision_at_k(y_test, leaky_scores, 50),
        },
    ]
)
print(f"Client-held-out split seed: {split_seed}; train/test clients are disjoint.")
print("Same held-out rows and base rate are used in both runs.")
display(leakage_results)
print(
    "The future_change_ratio result is invalid by construction: it uses the label period and defines the threshold. "
    "Do not report it as model performance."
)

honest_average_precision = float(leakage_results.loc[0, "average_precision"])
del leaky_model, leaky_train, leaky_test, leaky_scores, leak_ratio, feature_rows
print(f"Retained clean-feature average precision for this split: {honest_average_precision:.3f}")
print("The only retained model score is from features available by the March 24 decision.")

Client-held-out split seed: 42; train/test clients are disjoint.
Same held-out rows and base rate are used in both runs.


,feature_set,test_pages,test_clients,target_base_rate,average_precision,precision_at_50
0,Clean: five pre-decision features,19901,11,0.271846,0.339916,0.12
1,INVALID: future_change_ratio leakage,19901,11,0.271846,0.997039,1.00


The future_change_ratio result is invalid by construction: it uses the label period and defines the threshold. Do not report it as model performance.
Retained clean-feature average precision for this split: 0.340
The only retained model score is from features available by the March 24 decision.


## 4. What I excluded and why

- `trend_direction` and `trend_pct`: starter label sources in the separate 90-day snapshot; they do not define this future-window target and are not candidate features.
- `future_change_ratio`, `reference_avg_daily_impressions`, `outcome_avg_daily_impressions`, and `future_decline`: label construction or outcome-period fields; never model inputs.
- `client_hash_id` and `content_hash_id`: pseudonymous context for grouping and row grain only; not predictors and not displayed.
- `report_date`: used only to select the March partition and date windows.
- `gsc_data_available`: used only as an explicit `IS TRUE` eligibility filter, not a feature.
- Product flags, composite scores, client names, domains, URLs, raw queries, and text: not used or present in the public feature frame.

**Population limitation:** requiring at least three observed days in both reference and outcome windows depends on outcome-window coverage. The reported result therefore applies only to pages with that coverage; it must not be generalized to all content or described as future availability known at the decision moment.

In [4]:
excluded_field_audit = pd.DataFrame(
    [
        {"field": "trend_direction / trend_pct", "role": "label source for the starter proxy", "decision": "excluded; not part of the future-window task"},
        {"field": "future_change_ratio / outcome average", "role": "future outcome and label-derived", "decision": "deliberate leakage challenge only; deleted after comparison"},
        {"field": "future_decline", "role": "target", "decision": "label only; never a feature"},
        {"field": "client_hash_id / content_hash_id", "role": "pseudonymous context", "decision": "grouping/grain only; not displayed or modeled"},
        {"field": "report_date", "role": "window selector", "decision": "used to construct date windows only"},
        {"field": "gsc_data_available", "role": "coverage flag", "decision": "filter with IS TRUE; not modeled"},
        {"field": "product flags / composite scores", "role": "decision-derived", "decision": "not included in the feature frame"},
        {"field": "client names / domains / URLs / queries / raw text", "role": "private or unavailable", "decision": "not loaded into the analysis frame or displayed"},
    ]
)
display(excluded_field_audit)

assert set(FEATURE_COLUMNS).isdisjoint(
    {
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_data_available",
        "trend_direction",
        "trend_pct",
        "future_change_ratio",
        "future_decline",
    }
)
assert set(feature_frame.columns).isdisjoint({"client_hash_id", "content_hash_id"})
print("Feature and identifier exclusion assertions are defined above.")

,field,role,decision
0,trend_direction / trend_pct,label source for the starter proxy,excluded; not part of the future-window task
1,future_change_ratio / outcome average,future outcome and label-derived,deliberate leakage challenge only; deleted aft...
2,future_decline,target,label only; never a feature
3,client_hash_id / content_hash_id,pseudonymous context,grouping/grain only; not displayed or modeled
4,report_date,window selector,used to construct date windows only
5,gsc_data_available,coverage flag,filter with IS TRUE; not modeled
6,product flags / composite scores,decision-derived,not included in the feature frame
7,client names / domains / URLs / queries / raw ...,private or unavailable,not loaded into the analysis frame or displayed


Feature and identifier exclusion assertions are defined above.


## Self-check

Leave these unchecked until you run the notebook top to bottom, enter your read-only Hugging Face token through the hidden `getpass` prompt, and inspect the outputs yourself. Never put the token in notebook source or chat.

- [ ] Feature rows are one client-page each and use only March 1–23 inputs
- [ ] Label uses March 24–31 observed outcomes and is described as a proxy
- [ ] All five feature notes include meaning, missingness, and decision-time availability
- [ ] Client-held-out split has no overlap and both target classes
- [ ] Clean and deliberately leaky models use the same held-out rows and report the base rate
- [ ] Leaky result is identified as invalid and its temporary feature is discarded
- [ ] No client/page identifiers, names, domains, URLs, or private queries appear in output
- [ ] Notebook runs top to bottom; metric values and limits are reviewed before use